In [ ]:
import json
import os
from datetime import datetime
from typing import Any, Dict, Optional

import cv2


IMG_FORMATS = {
    ".jpg",
    ".jpeg",
    ".png"
}


class YoloToAzureCocoConverter:
    """
    Converts a YOLO annotation dataset to Azure COCO format.
    """

    def __init__(
        self,
        images_folder: str,
        output_file: str,
        datastore_name: str,
        path_on_datastore: str,
        storage_account_name: str,
        categories: Dict[int, str],
        yolo_to_azure_category_map: Dict[int, int],
        labels_folder: Optional[str] = None,
        min_conf: float = 0.0,
    ):
        """
        Parameters
        ----------
        images_folder: str
            Path to the folder containing the images.
        output_file: str
            Path to the JSON file where the Azure COCO annotations will be stored.
        datastore_name: str
            Name of the datastore to be used in the COCO file URLs.
        image_storage_account: str
            Name of the storage account to be used in the COCO file URLs.
        categories: Dict[int, str]
            Category IDs to names.
        yolo_to_azure_category_map: Dict[int, int]
            Mapping from YOLO category IDs to Azure Data Labelling category IDs.
        labels_folder: Optional[str] = None
            Optional: folder containing the annotations. If omitted, annotations
            are assumed to be in the images_folder.
        min_conf: float = 0.0
            Confidence threshold for annotations.
        """
        if os.path.splitext(output_file)[1] != ".json":
            raise ValueError(f"Output file should be a .json file.")

        self.images_folder = images_folder
        self.output_file = output_file
        self.datastore_name = datastore_name
        self.path_on_datastore = path_on_datastore
        self.storage_account_name = storage_account_name
        self.labels_folder = labels_folder if labels_folder is not None else images_folder
        self.categories = categories
        self.yolo_to_azure_category_map = yolo_to_azure_category_map
        self.min_conf = min_conf
        self.coco_json = {
            "images": [],
            "annotations": [],
            "categories": self.categories,
        }
        self.annotation_id = 1

    def _yolo_to_coco(
        self, yolo_annotation: str, img_width: int, img_height: int
    ) -> Dict[str, Any]:
        """
        Converts a single YOLO annotation to COCO format.

        Parameters
        ----------
        yolo_annotation: str
            One line of YOLO annotation.
        img_width: int
            Width of the image.
        img_height: int
            Height of the image.

        Returns
        -------
        dict
            The annotation in COCO format.
        """
        # Split the annotation and select only the first 5 values
        values = yolo_annotation.split()

        if len(values) >= 6:
            class_id, x_center, y_center, width, height, conf = map(float, values[:6])
        elif len(values) == 5:
            class_id, x_center, y_center, width, height = map(float, values)
            conf = 1.0
        else:
            raise ValueError(f"Unrecognized annotation string format: {yolo_annotation}")

        coco_class_id = self.yolo_to_azure_category_map[class_id]
        x_center, y_center, width, height = (
            x_center * img_width,
            y_center * img_height,
            width * img_width,
            height * img_height,
        )
        # Normalize bbox (COCO format expects top left x, top left y, width, height)
        x_min = (x_center - width / 2) / img_width
        y_min = (y_center - height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height
        # Azure COCO expects bbox to be rounded to 17 decimal places
        # and area to be rounded to 10 decimal places
        bbox = [
            round(x_min, 17),
            round(y_min, 17),
            round(norm_width, 17),
            round(norm_height, 17),
        ]
        area = round(norm_width * norm_height, 10)

        return {"category_id": int(coco_class_id), "bbox": bbox, "area": area, "conf": conf}

    def convert(self):
        """
        Converts the YOLO annotations in the input folder to Azure COCO format and saves them in the output folder.
        """
        img_files = [
            file for file in os.listdir(self.images_folder)
            if os.path.splitext(file)[1].lower() in IMG_FORMATS
        ]

        for img_file in img_files:
            image_id = len(self.coco_json["images"]) + 1
            img = cv2.imread(os.path.join(self.images_folder, img_file))
            height, width, _ = img.shape

            file_name_formatted = os.path.join(self.path_on_datastore, img_file)
            coco_url = f"AmlDatastore://{self.datastore_name}/{file_name_formatted}"
            absolute_url = f"https://{self.storage_account_name}.blob.core.windows.net/{self.datastore_name}/{file_name_formatted.replace(' ', '%20')}"

            self.coco_json["images"].append(
                {
                    "id": image_id,
                    "width": width,
                    "height": height,
                    "file_name": file_name_formatted,
                    "coco_url": coco_url,
                    "absolute_url": absolute_url,
                    "date_captured": datetime.now().strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
                }
            )

            annotation_file = os.path.join(
                self.labels_folder,
                os.path.splitext(os.path.basename(img_file))[0] + ".txt",
            )

            if os.path.exists(annotation_file):
                with open(annotation_file, "r") as f:
                    for line in f:
                        coco_annotation = self._yolo_to_coco(
                            line, width, height
                        )
                        if coco_annotation["conf"] >= self.min_conf:
                            # Construct annotation dict with the specified key order and formatted values
                            formatted_annotation = {
                                "id": self.annotation_id,
                                "category_id": coco_annotation["category_id"],
                                "image_id": image_id,
                                "area": coco_annotation["area"],
                                "bbox": coco_annotation["bbox"],
                            }
                            self.coco_json["annotations"].append(formatted_annotation)
                            self.annotation_id += 1

        with open(self.output_file, "w") as f:
            json.dump(self.coco_json, f, indent=4)

In [ ]:
images_folder = "../datasets/experiments/zwerfafval/data_inwinning_260713/frames_selected_200/images/"
labels_folder = "../datasets/experiments/zwerfafval/data_inwinning_260713/frames_selected_200/labels/"

output_file = "../datasets/experiments/zwerfafval/annotatieproject/testset_260713.json"

datastore_name = "experiments"
path_on_datastore = "zwerfafval/annotatieproject/testset_260713/"
storage_account_name = "cvodataweupgwapeg4pyiw5e"

confidence_threshold = 0.25

categories = [
    {
        "id": 1,
        "name": "zwerfafval_grof",
        "supercategory": "none"
    },
    {
        "id": 4,
        "name": "zwerfafval_fijn",
        "supercategory": "none"
    }
]
yolo_to_azure_category_map = {
    0: 1,
    1: 4
}

converter = YoloToAzureCocoConverter(
    images_folder=images_folder,
    output_file=output_file,
    datastore_name=datastore_name,
    path_on_datastore=path_on_datastore,
    storage_account_name=storage_account_name,
    categories=categories,
    yolo_to_azure_category_map=yolo_to_azure_category_map,
    labels_folder=labels_folder,
    min_conf=confidence_threshold
)

converter.convert()